# Yearly Sunspot Forecasting with Prophet

In [ ]:

import pandas as pd
from prophet import Prophet  # Ensure 'prophet' is installed locally
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

# Load and preprocess the yearly dataset
file_path = 'SN_y_tot_V2.0.csv'
data = pd.read_csv(file_path, delimiter=';', header=None)

# Assign meaningful column names based on the description in the project PDF
if "yearly" == "daily":
    data.columns = ['Year', 'Month', 'Day', 'Date_Fraction', 'Sunspot_Count', 'Std_Dev', 'Observations', 'Provisional']
    data['Date'] = pd.to_datetime(data[['Year', 'Month', 'Day']])
elif "yearly" == "monthly":
    data.columns = ['Year', 'Month', 'Date_Fraction', 'Sunspot_Count', 'Std_Dev', 'Observations', 'Provisional']
    data['Date'] = pd.to_datetime(data[['Year', 'Month']].assign(Day=1))
elif "yearly" == "yearly":
    data.columns = ['Year', 'Date_Fraction', 'Sunspot_Count', 'Std_Dev', 'Observations', 'Provisional']
    data['Date'] = pd.to_datetime(data[['Year']].assign(Month=1, Day=1))

# Select necessary columns for FBProphet and handle missing values (-1)
data = data[['Date', 'Sunspot_Count']]
data = data[data['Sunspot_Count'] != -1]

# Rename columns to match Prophet's requirements
data = data.rename(columns={'Date': 'ds', 'Sunspot_Count': 'y'})

# Initialize the Prophet model
prophet = Prophet()

# Fit the model
prophet.fit(data)

# Make future predictions
future = prophet.make_future_dataframe(periods=20, freq='Y')
forecast = prophet.predict(future)

# Plot the forecast
fig = prophet.plot(forecast)
plt.title('Yearly Sunspot Forecast (Prophet)')
plt.show()

# Evaluate the model
test_start_idx = len(data) - 20
train = data.iloc[:test_start_idx]
test = data.iloc[test_start_idx:]

prophet.fit(train)
forecast_test = prophet.predict(test[['ds']])

# Calculate evaluation metrics
mae = mean_absolute_error(test['y'], forecast_test['yhat'])
mape = mean_absolute_percentage_error(test['y'], forecast_test['yhat'])
r2 = r2_score(test['y'], forecast_test['yhat'])

# Display metrics
print('Evaluation Results:')
print(f'Mean Absolute Error: {mae}')
print(f'Mean Absolute Percentage Error: {mape}')
print(f'R² Score: {r2}')
